In [ ]:
#| default_exp main01

## Chat UI (DaisyUI)

Pure [DaisyUI](https://daisyui.com/) — **no MonsterUI / FrankenUI**.
MonsterUI already loads DaisyUI under the hood, so swapping headers alone looks identical.
This notebook uses a different theme (`nord`), navbar, a Chat/Tables taskbar under the input, a viz-picker pill on the message box, and chat avatars so the difference is obvious.

Messages go to the in-process `string_therapy` `/controller` (editable sibling at `../string_therapy`): browse category/endpoint **strings**, or route+forward when the user is ready (asks for missing required params).
ZINC query result tables load into the **Tables** tab (`table_workspace` + `ui/table_client.js`); a badge on the Tables icon marks new loads.

In [ ]:
#| export
import json
import os
import uuid
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path

from dotenv import load_dotenv
from fasthtml.common import *
from starlette.testclient import TestClient
from string_therapy import create_app
from string_therapy_for_material_science.table_workspace import table_workspace

REPO = Path(__file__).resolve().parents[1]
if not (REPO / ".env").exists() and (REPO.parent / ".env").exists():
    REPO = REPO.parent
load_dotenv(REPO / ".env")

# Local UI talks to an in-process controller (no Bearer header).
# Override in .env if you need real JWT auth.
if not (os.getenv("JWT_AUTH_DISABLED") or "").strip():
    os.environ["JWT_AUTH_DISABLED"] = "true"

_controller = TestClient(create_app())

DAISY_THEME = "nord"

daisy_hdrs = (
    Script(src="https://cdn.tailwindcss.com"),
    Link(
        rel="stylesheet",
        href="https://cdn.jsdelivr.net/npm/daisyui@4.12.22/dist/full.min.css",
        type="text/css",
    ),
    Script(src="https://unpkg.com/lucide@0.468.0"),
    Script(src="https://cdn.plot.ly/plotly-2.35.2.min.js"),
    MarkdownJS(".chat-bubble.marked"),
    # Layout-only: Plotly/flex need fixed height; everything else is DaisyUI/Tailwind classes.
    # `.marked` chat bubbles get light markdown typography (lists, code, paragraphs).
    Style("""
      html, body { height: 100%; margin: 0; }
      #main-panel { min-height: 0; }
      [id$="-chart"] { width: 100%; height: 100%; min-height: 320px; }
      .chat-bubble.marked { white-space: normal; line-height: 1.45; }
      .chat-bubble.marked > :first-child { margin-top: 0; }
      .chat-bubble.marked > :last-child { margin-bottom: 0; }
      .chat-bubble.marked p { margin: 0.4em 0; }
      .chat-bubble.marked ul, .chat-bubble.marked ol {
        margin: 0.4em 0; padding-left: 1.25rem; list-style-position: outside;
      }
      .chat-bubble.marked ul { list-style-type: disc; }
      .chat-bubble.marked ol { list-style-type: decimal; }
      .chat-bubble.marked li { margin: 0.2em 0; }
      .chat-bubble.marked li > p { margin: 0.15em 0; }
      .chat-bubble.marked code {
        font-size: 0.9em; padding: 0.1em 0.35em; border-radius: 0.25rem;
        background: color-mix(in oklab, currentColor 12%, transparent);
      }
      .chat-bubble.marked pre {
        margin: 0.5em 0; padding: 0.5em 0.65em; border-radius: 0.4rem;
        overflow-x: auto;
        background: color-mix(in oklab, currentColor 10%, transparent);
      }
      .chat-bubble.marked pre code { padding: 0; background: transparent; }
      .chat-bubble.marked strong { font-weight: 700; }
      .chat-bubble.marked a { text-decoration: underline; }
    """),
)


# %% ../nbs/02_main.ipynb #ea4fddae-daisy-0003


## Icons and state

Taskbar icons, shared paths, and the in-memory Graph table store.


In [ ]:
#| export
class IconKind(Enum):
    """Taskbar icon category: interface chrome vs IO visualization."""
    INTERFACE = "interface"
    IO = "io"

@dataclass
class Icon:
    """One taskbar or IO picker icon (id, label, lucide name, kind)."""
    id: str
    label: str
    icon: str
    kind: IconKind
    active: bool = False
    extra: dict = field(default_factory=dict)

interface_icons = [
    Icon("chat", "Chat", "message-circle", IconKind.INTERFACE, active=True),
    Icon("graph", "Graph", "workflow", IconKind.INTERFACE),
]

GRAPH_JSON = REPO / "routes" / "router_graph.json"

# One shared IO visualization slot — settings / router payloads land here.
# Icons in the picker are built dynamically (category click or chat), not a fixed set.
# Graph tab holds query result tables (see table_workspace + ui/table_client.js).
_VIEW_IDS = ("chat-view", "graph-view", "io-view")
_ACTIVE_OUTLINE_CLASSES = ("ring-2", "ring-primary", "text-primary")
DEFAULT_VIZ_URL = ""

VIZ_CLIENT_JS = REPO / "ui" / "viz_client.js"
TABLE_CLIENT_JS = REPO / "ui" / "table_client.js"
CHAT_CLIENT_JS = REPO / "ui" / "chat_client.js"

# session_id → table_id → {title, twin, columns, rows}
_TABLE_STORE: dict[str, dict[str, dict]] = {}


def _store_table(session_id: str, entry: dict) -> None:
    """Persist a Graph-tab table entry for ``session_id`` (upsert by ``entry["id"]``)."""
    sid = (session_id or "").strip() or "_anon"
    tid = str(entry.get("id") or "").strip()
    if not tid:
        raise ValueError("table id required")
    rows = entry.get("rows") or []
    if not isinstance(rows, list) or not rows:
        raise ValueError("rows required")
    bucket = _TABLE_STORE.setdefault(sid, {})
    bucket[tid] = {
        "id": tid,
        "title": str(entry.get("title") or entry.get("twin") or tid),
        "twin": str(entry.get("twin") or ""),
        "columns": list(entry.get("columns") or []),
        "rows": rows,
    }


def _get_table(session_id: str, table_id: str) -> dict | None:
    """Return a stored table dict for ``session_id``/``table_id``, or ``None``."""
    sid = (session_id or "").strip() or "_anon"
    tid = (table_id or "").strip()
    if not tid:
        return None
    return (_TABLE_STORE.get(sid) or {}).get(tid)


## Layout helpers

Navbar, taskbar, and view-switch JS helpers.


In [ ]:
#| export
def lucide_icon(name: str, cls: str = "w-5 h-5"):
    """Return a Lucide ``<i data-lucide=...>`` element for client-side icon hydration."""
    return I(data_lucide=name, cls=cls)

def _show_view_js(view_id: str) -> str:
    """JS snippet that hides all main views and shows ``view_id``."""
    parts = [f"document.getElementById('{v}').classList.add('hidden')" for v in _VIEW_IDS]
    parts.append(f"document.getElementById('{view_id}').classList.remove('hidden')")
    return "; ".join(parts)

def _set_active_icon_js(icon_id: str, *, attr: str = "data-taskbar-icon") -> str:
    """JS snippet that outlines the active taskbar/IO icon matching ``icon_id``."""
    remove = "".join(f"b.classList.remove('{c}');" for c in _ACTIVE_OUTLINE_CLASSES)
    add = "".join(f"t.classList.add('{c}');" for c in _ACTIVE_OUTLINE_CLASSES)
    return (
        f"document.querySelectorAll('[{attr}]').forEach(b=>{{"
        + remove
        + "});"
        + f"const t=document.querySelector('[{attr}=\"{icon_id}\"]');"
        + f"if(t){{{add}}}"
    )

def _view_for_icon(icon_id: str) -> str:
    """Map a taskbar icon id to its main-panel view element id."""
    return {"chat": "chat-view", "graph": "graph-view"}.get(icon_id, "chat-view")

def top_navbar():
    """Top brand bar for the DaisyUI shell."""
    return Div(
        Div(
            A(
                lucide_icon("sparkles", cls="w-5 h-5"),
                Span("String Therapy", cls="font-semibold tracking-tight"),
                cls="btn btn-ghost text-xl gap-2",
            ),
            cls="flex-1",
        ),
        cls="navbar bg-base-100/80 backdrop-blur border-b border-base-300 px-4 shadow-sm",
    )

def _taskbar_btn(ic: Icon):
    """Build one taskbar button that switches views and updates the active outline."""
    outline = " ".join(_ACTIVE_OUTLINE_CLASSES) if ic.active else ""
    icon_el = lucide_icon(ic.icon, cls="w-4 h-4")
    if ic.id == "graph":
        icon_el = Div(
            lucide_icon(ic.icon, cls="w-4 h-4"),
            Span(
                "",
                id="graph-badge",
                cls=(
                    "badge badge-primary badge-xs absolute -top-1.5 -right-2 "
                    "hidden min-w-[1.1rem] h-4 px-1 leading-none"
                ),
            ),
            cls="relative inline-flex",
        )
    clear_badge = "if(window.__stClearTableBadge)window.__stClearTableBadge();" if ic.id == "graph" else ""
    return Button(
        icon_el,
        Span(ic.label, cls="text-[10px] leading-tight"),
        type="button",
        title=ic.label,
        id=f"taskbar-icon-{ic.id}",
        data_taskbar_icon=ic.id,
        cls=f"btn btn-ghost btn-sm h-auto py-1.5 px-3 gap-1 flex-col {outline}".strip(),
        onclick="; ".join([
            _set_active_icon_js(ic.id),
            _show_view_js(_view_for_icon(ic.id)),
            clear_badge,
            "lucide.createIcons();",
        ]),
    )

def taskbar():
    """Bottom taskbar with Chat / Graph interface icons."""
    return Div(
        *[_taskbar_btn(ic) for ic in interface_icons],
        id="taskbar",
        cls="flex items-center justify-center gap-1 mt-2 px-1 py-1 rounded-xl bg-base-100/90 border border-base-300 shadow-sm",
    )

def viz_picker():
    """Filled dynamically by viz_client.js when a category opens or chat returns a viz."""
    return Div(
        id="viz-picker",
        cls=(
            "absolute left-1/2 -translate-x-1/2 -top-3 z-10 "
            "inline-flex flex-wrap items-center justify-center gap-0.5 "
            "max-w-xl px-1.5 py-0.5 "
            "rounded-full border border-base-300 bg-base-100 shadow-sm "
            "hidden"
        ),
    )


## Views

Chat, Graph (tables), IO visualization, and the fixed input bar.


In [ ]:
#| export
def chat_bubble(role: str, content: str):
    """Render a single user or assistant chat bubble with avatar."""
    is_user = role == "user"
    avatar = Div(
        Div(
            lucide_icon("user" if is_user else "bot", cls="w-4 h-4"),
            cls="w-10 rounded-full bg-primary text-primary-content grid place-items-center"
            if is_user
            else "w-10 rounded-full bg-secondary text-secondary-content grid place-items-center",
        ),
        cls="chat-image avatar",
    )
    bubble_cls = (
        f"chat-bubble max-w-xl shadow "
        f"{'chat-bubble-primary whitespace-pre-line' if is_user else 'chat-bubble-secondary marked'}"
    )
    return Div(
        avatar,
        Div("You" if is_user else "Router", cls="chat-header opacity-70 text-xs mb-1"),
        Div(content, cls=bubble_cls),
        cls=f"chat {'chat-end' if is_user else 'chat-start'} px-2",
    )

def oob_chat_bubble(role: str, content: str, *, after_js: str = ""):
    """OOB append to #chat-messages. Optional after_js runs in the same OOB (HTMX executes it)."""
    js = "lucide.createIcons();" + (after_js or "")
    return Div(
        chat_bubble(role, content),
        Script(js),
        hx_swap_oob="beforeend:#chat-messages",
    )

def chat_area():
    """Scrollable chat view that hosts #chat-messages."""
    return Div(
        Div(
            Div(
                Span("Ask the router", cls="font-medium"),
                cls="mb-4 px-2",
            ),
            Div(id="chat-messages", cls="flex flex-col gap-3 w-full"),
            cls="w-full max-w-2xl mx-auto py-6 px-3",
        ),
        id="chat-view",
        cls="flex flex-col flex-1 min-h-0 overflow-y-auto",
    )

def graph_view():
    """Tables workspace — query-agent row results land here (multi-tab)."""
    return Div(
        table_workspace(),
        id="graph-view",
        cls="flex flex-col flex-1 min-h-0 overflow-hidden hidden",
    )

def io_viz_view():
    """Single visualization surface — every LangGraph figure lands here."""
    return Div(
        Div(
            Span("Visualization", id="io-title", cls="font-medium"),
            Span("", id="io-badge", cls="badge badge-ghost badge-sm ml-2"),
            cls="px-4 py-2 border-b border-base-300 bg-base-100/70 flex items-center",
        ),
        Div(
            P(
                "Ask for a chart or open a category in the graph",
                id="io-status",
                cls="text-sm opacity-60 px-3 pt-3",
            ),
            Div(id="io-chart", cls="w-full flex-1 min-h-0"),
            cls="flex flex-col flex-1 min-h-0",
        ),
        id="io-view",
        data_viz_url=DEFAULT_VIZ_URL,
        data_viz_method="post",
        data_viz_intent="",
        cls="flex flex-col flex-1 min-h-0 overflow-hidden hidden",
    )

def main_panel():
    """Stack of chat, graph, and IO views (only one visible at a time)."""
    return Div(
        chat_area(),
        graph_view(),
        io_viz_view(),
        id="main-panel",
        cls="flex flex-col flex-1 min-h-0 overflow-hidden pb-40",
    )

def input_bar():
    """Fixed bottom composer: message form, viz picker slot, and taskbar."""
    return Div(
        Form(
            Input(type="hidden", name="attached_table_id", id="attached-table-id", value=""),
            Div(
                id="table-ref-chip",
                cls="mb-1.5 min-h-0 flex flex-wrap items-center gap-1 hidden",
            ),
            Div(
                viz_picker(),
                Textarea(
                    placeholder="Message the router… (click a Graph table to attach it)",
                    name="message",
                    rows=2,
                    cls="textarea textarea-bordered textarea-primary w-full resize-none rounded-2xl pt-6 pr-14 leading-snug",
                    id="chat-input",
                    onkeydown="if(event.key==='Enter'&&!event.shiftKey){event.preventDefault();this.form.requestSubmit()}",
                    hx_on_input="""
                        let btn = document.getElementById('submit-btn');
                        if (this.value.trim().length > 0) btn.classList.remove('btn-disabled');
                        else btn.classList.add('btn-disabled');
                        """,
                ),
                Button(
                    lucide_icon("send", cls="w-4 h-4"),
                    id="submit-btn",
                    cls="btn btn-primary btn-circle btn-sm absolute right-3 bottom-3 btn-disabled",
                    type="submit",
                ),
                cls="relative w-full",
            ),
            hx_post="/chat/send",
            hx_target="#htmx-sink",
            hx_swap="innerHTML",
            hx_on__before_request="""
                const ta = document.getElementById('chat-input');
                const msg = (ta && ta.value || '').trim();
                if (!msg || document.getElementById('chat-loading')) {
                  event.preventDefault();
                  return;
                }
                if (window.__stAppendOutgoingChat) window.__stAppendOutgoingChat(msg);
                ta.value = '';
                document.getElementById('submit-btn').classList.add('btn-disabled');
                """,
            hx_on__after_request=(
                "if(window.__stRemoveChatLoading)window.__stRemoveChatLoading();"
                "document.getElementById('submit-btn').classList.add('btn-disabled');"
                "if(window.__stScrollChatToBottom)window.__stScrollChatToBottom();"
                "lucide.createIcons();"
            ),
            cls="w-full",
        ),
        taskbar(),
        id="input-bar",
        cls="fixed bottom-0 left-0 right-0 z-10 px-3 pb-3 pt-5 max-w-2xl mx-auto w-full",
    )


## App and static routes

FastHTML app, JS assets, table upsert API, and the shell page.


In [ ]:
#| export
app, rt = fast_app(
    hdrs=daisy_hdrs,
    pico=False,
    htmlkw={"data-theme": DAISY_THEME},
    bodykw={"class": "min-h-screen bg-base-200"},
)

@rt("/ui/viz_client.js")
def viz_client_js():
    """Serve ``ui/viz_client.js`` for the IO visualization client."""
    return FileResponse(VIZ_CLIENT_JS, media_type="application/javascript")

@rt("/ui/table_client.js")
def table_client_js():
    """Serve ``ui/table_client.js`` for the Graph tables workspace."""
    return FileResponse(TABLE_CLIENT_JS, media_type="application/javascript")

@rt("/ui/chat_client.js")
def chat_client_js():
    """Serve ``ui/chat_client.js`` for outgoing chat UX helpers."""
    return FileResponse(CHAT_CLIENT_JS, media_type="application/javascript")


@rt("/api/tables", methods=["POST"])
async def api_tables_upsert(req, session):
    """Store Graph-tab rows for the current session (used by the table analyst)."""
    if "session_id" not in session:
        session["session_id"] = str(uuid.uuid4())
    try:
        body = await req.json()
    except Exception:  # noqa: BLE001
        return JSONResponse({"status": "error", "detail": "Invalid JSON"}, status_code=400)
    if not isinstance(body, dict):
        return JSONResponse({"status": "error", "detail": "JSON object required"}, status_code=400)
    try:
        _store_table(session["session_id"], body)
    except ValueError as e:
        return JSONResponse({"status": "error", "detail": str(e)}, status_code=400)
    return JSONResponse(
        {
            "status": "ok",
            "id": body.get("id"),
            "row_count": len(body.get("rows") or []),
        }
    )


@rt("/health")
def health():
    """Liveness for Railway / deploy checks."""
    return JSONResponse({"status": "ok"})


@rt("/")
def get(session):
    """Render the full-page shell (navbar, main panel, input bar, client scripts)."""
    if "session_id" not in session:
        session["session_id"] = str(uuid.uuid4())

    return Div(
        top_navbar(),
        main_panel(),
        input_bar(),
        Div(id="htmx-sink", cls="hidden", aria_hidden="true"),
        Script(src="/ui/viz_client.js"),
        Script(src="/ui/table_client.js"),
        Script(src="/ui/chat_client.js"),
        Script("lucide.createIcons();"),
        cls="flex flex-col h-screen",
    )


## Chat

Controller / analyst dispatch. Bubble text prefers downstream `response`; side effects open viz or tables.


In [ ]:
#| export
def _chat_reply_for_parsed(parsed: dict | None, raw_reply: str) -> str:
    """Prefer downstream prose; avoid dumping large JSON payloads into the bubble."""
    if isinstance(parsed, dict) and parsed.get("status") == "error":
        err = (
            parsed.get("error")
            or parsed.get("detail")
            or parsed.get("response")
            or raw_reply
            or "unknown error"
        )
        hint = parsed.get("hint")
        return str(err) + (f" — {hint}" if hint else "")
    reply = raw_reply if isinstance(raw_reply, str) else json.dumps(raw_reply, indent=2)
    if len(reply) > 500 and (reply.lstrip().startswith("{") or reply.lstrip().startswith("[")):
        return "Request completed."
    return reply


def _parsed_has_viz(parsed: dict | None) -> bool:
    """True when ``parsed`` carries a plotly/plot/panel payload for the IO view."""
    return isinstance(parsed, dict) and bool(
        parsed.get("plotly") or parsed.get("plot") or parsed.get("panel")
    )


def _parsed_has_table(parsed: dict | None) -> bool:
    """True when query-agent (or similar) returned a non-empty row list."""
    if not isinstance(parsed, dict):
        return False
    rows = parsed.get("rows")
    return isinstance(rows, list) and len(rows) > 0 and isinstance(rows[0], dict)


@rt("/chat/send", methods=["POST"])
async def chat_send(message: str, session, attached_table_id: str = ""):
    """Handle chat submit via controller; attach table hold context when a Graph tab is selected."""
    user_msg = message.strip()
    if not user_msg:
        return ""

    if "session_id" not in session:
        session["session_id"] = str(uuid.uuid4())
    sid = session.get("session_id", "").strip()
    parsed: dict | None = None
    reply = ""
    data: dict = {}
    table_id = (attached_table_id or "").strip()

    payload: dict = {"message": user_msg, "use_history": True, "response_format": "json"}
    if sid:
        payload["session_id"] = sid

    if table_id:
        entry = _get_table(sid, table_id)
        if not entry:
            parsed = {
                "status": "error",
                "error": "Attached table not found on the server. Re-run the query or re-attach the tab.",
                "visualization": "table_analyst",
                "endpoint": "plotly",
                "source": "analyst",
            }
            reply = _chat_reply_for_parsed(parsed, "")
        else:
            payload["table"] = {
                "id": entry.get("id"),
                "title": entry.get("title"),
                "twin": entry.get("twin"),
                "columns": entry.get("columns") or [],
                "rows": entry.get("rows") or [],
            }

    if not reply:
        r = _controller.post("/controller", json=payload)
        try:
            data = r.json()
        except Exception:  # noqa: BLE001
            data = {"detail": r.text[:2000]}

        parsed = data.get("parsed") if isinstance(data, dict) else None
        if r.status_code >= 400:
            reply = data.get("detail") or data.get("error") or f"Controller error HTTP {r.status_code}"
            parsed = None
        else:
            reply = _chat_reply_for_parsed(
                parsed if isinstance(parsed, dict) else None,
                data.get("response") or data.get("detail") or "No response",
            )

        new_sid = (data.get("session_id") or sid or "").strip()
        if new_sid:
            session["session_id"] = new_sid

    # Run viz / table open inside the assistant OOB bubble — that swap executes scripts.
    after_js = "window.__stLastChatMessage = " + json.dumps(user_msg) + ";"
    if _parsed_has_viz(parsed):
        after_js += (
            "setTimeout(function(){ window.openIoVizFromChat(" + json.dumps(parsed) + "); }, 0);"
        )
    if _parsed_has_table(parsed):
        # Drop bulky excel from chat bubble if somehow duplicated; keep for download in Tables.
        table_payload = {
            k: parsed[k]
            for k in ("twin_name", "row_count", "rows", "excel", "status", "response")
            if isinstance(parsed, dict) and k in parsed
        }
        after_js += (
            "setTimeout(function(){ window.loadTableFromChat(" + json.dumps(table_payload) + "); }, 0);"
        )

    # User bubble + loading were already painted client-side; only append the reply.
    after_js = (
        "if(window.__stRemoveChatLoading)window.__stRemoveChatLoading();" + after_js
    )
    parts: list = [
        oob_chat_bubble("assistant", reply, after_js=after_js),
        Div(id="htmx-sink"),  # primary hx-swap target (innerHTML)
    ]

    return tuple(parts)

if __name__ == "__main__":
    # Default :4546 on this host (Railway Caddy proxies here). Set UVICORN_RELOAD=1 for dev.
    serve(
        appname="string_therapy_for_material_science.main01",
        port=int(os.getenv("PORT", "4546")),
        reload=(os.getenv("UVICORN_RELOAD", "0").strip() == "1"),
    )


## One-time DB setup (optional)

Run when you need to (re)bootstrap schema + seed the router graph. Not exported into `main01`.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from string_therapy import ensure_schema, seed_router_db

repo = Path.cwd() if (Path.cwd() / '.env').exists() else Path.cwd().parent
load_dotenv(repo / '.env')

ensure_schema()
seed_router_db(repo / 'routes' / 'router_graph.json')
print('schema + seed ok')